# Assistente dizionario codici — banco di prova

Notebook autonomo e volutamente **semplice**: non classifica, non indovina, non ha punteggi né
alias. Prova **tutte le combinazioni di regole** sui dati veri e conta quanti codici combaciano.

- **Codici articolo**: si prendono i **codici univoci** presenti negli XML, tutti insieme, **senza
  dividerli per fattura** (e senza guardare `CFOR` per decidere). Per ogni `CodiceTipoXML` si provano
  i 3 campi canonici (`EAN`, `CODICE_CLIENTE`, `CODICE_FORNITORE`) × le 7 regole lato XML × le 7
  regole lato carico: si normalizzano i codici XML con la prima regola, i valori di anagrafica con la
  seconda, e si conta quanti codici si ritrovano. Vince la combinazione con più match.
- **DDT**: qui si ragiona **dentro ogni fattura** (`CEAS + numero fattura`), altrimenti lo stesso
  numero di bolla di due fatture diverse si incrocerebbe a caso. Per ogni fattura si provano le
  7 × 7 regole e si conta quanti `NumeroDDT` dell'XML trovano un `NDOGIAB` della **stessa** fattura.

Le 7 regole sono **esattamente** quelle del registro `REGOLE` di `Quadratura_Fatture.ipynb`
(cella `[4]`), pulizia di base compresa: la prova usa quindi la stessa normalizzazione del processo.

A parità di match si preferisce la regola meno "distruttiva" (prima `RIMUOVI_SEPARATORI`, poi le
altre): la griglia con tutte le combinazioni resta comunque disponibile per scegliere a mano.

Output: `export/proposte_dizionario_codici.xlsx` con 4 fogli — `ProposteCodici`, `ProposteDDT`,
`GrigliaCodici`, `GrigliaDDT`. **Il dizionario non viene mai letto né scritto**: le righe si copiano
a mano dopo averle guardate.

Eseguire le celle in ordine. Dettagli in [`docs/ASSISTENTE_DIZIONARIO.md`](docs/ASSISTENTE_DIZIONARIO.md).

## [1] Configurazione e regole

Percorsi (gli stessi della quadratura), `MODALITA_LOCALE` per girare fuori da Fabric, e il registro
delle 7 regole copiato dalla cella `[4]` del notebook principale. `normalizza(valore, regola)` è
l'unica funzione usata dal banco di prova.

In [ ]:
# ==============================================================
# [1] CONFIGURAZIONE E REGOLE
# ==============================================================

import os
import unicodedata
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import PurePosixPath

import pandas as pd


try:
    display
except NameError:
    def display(*oggetti, **kwargs):
        for oggetto in oggetti:
            print(oggetto)


# Stessa cartella OneLake del notebook di quadratura.
cartella = (
    "abfss://d072e408-7302-41de-a976-f6b6b626177e"
    "@onelake.dfs.fabric.microsoft.com/"
    "401c1a6a-fa86-43ab-a153-01e4fa9de79f/Files"
)

# True -> i file si leggono da CARTELLA_LOCALE, senza notebookutils.
MODALITA_LOCALE = False
CARTELLA_LOCALE = "./dati"

ZIP_FILE = "xml.zip"
zip_locale_python = "/tmp/assistente_prove_xml.zip"
zip_locale_fs = "file:" + zip_locale_python

CARICHI_FILE = "TAB_CM_Fatture_BI.xlsx"
CARICHI_SHEET = "TAB_CM_Fatture_BI"
EAN_FILE = "TAB_ANAGRAFICA_EAN.xlsx"
EAN_SHEET = "TAB_CM_ANAGRAFICA_EAN_BI"
CARF_FILE = "TAB_ANAGRAFICA_CARF.xlsx"
CARF_SHEET = "TAB_CM_ANAGRAFICA_CARF_BI"

ESPORTA_PROPOSTE = True
OUTPUT_FILE = "proposte_dizionario_codici.xlsx"

# Con SOLO_NUMERI "ART-002" diventa "2" e combacerebbe per caso
# con qualsiasi codice: nei codici articolo le chiavi di 1-2
# caratteri non vengono contate come match. I DDT corti esistono
# davvero ("H/1"), quindi per loro questo filtro non si applica.
MIN_LUNGHEZZA_CHIAVE = 3

# Campo canonico -> colonne da usare come riferimento.
CAMPI_RIFERIMENTO = {
    "EAN": [("EAN", "CEAN")],
    "CODICE_CLIENTE": [("CARICHI", "CART")],
    "CODICE_FORNITORE": [("CARF", "CARF"), ("CARICHI", "CALA")],
}


# --------------------------------------------------------------
# LE 7 REGOLE (copiate dalla cella [4] di Quadratura_Fatture)
# --------------------------------------------------------------

SEPARATORI = " -/._:"
CIFRE = "0123456789"


def pulito(valore):
    """Livello 1, sempre applicato: pulizia, trim, maiuscolo."""
    if valore is None or pd.isna(valore):
        return None
    if isinstance(valore, float) and valore.is_integer():
        valore = int(valore)
    testo = "".join(
        c for c in str(valore)
        if unicodedata.category(c) != "Cc"
    )
    testo = testo.strip().replace("\u00a0", "").upper()
    return testo or None


def _zeri(testo):
    senza = testo.lstrip("0")
    return "0" if senza == "" else senza


def _cifre(testo):
    return "".join(c for c in testo if c in CIFRE)


def _finalizza(testo):
    if testo is None:
        return None
    senza = "".join(c for c in testo if c not in SEPARATORI)
    if senza == "":
        return None
    return _zeri(senza) if _cifre(senza) == senza else senza


def _prima(testo):
    for indice, c in enumerate(testo):
        if not (c in CIFRE or "A" <= c <= "Z"):
            return testo[:indice]
    return testo


def _dividi(testo, separatori):
    parti = [testo]
    for separatore in separatori:
        parti = [
            pezzo
            for parte in parti
            for pezzo in parte.split(separatore)
        ]
    return parti


def _nessuna(testo):
    return testo


def _separatori(testo):
    return _finalizza(testo)


def _numeri(testo):
    cifre = _cifre(testo)
    return _zeri(cifre) if cifre else None


def _estrai_prima(testo):
    estratto = _prima(testo)
    return _finalizza(estratto or None)


def _estrai_ultima(testo):
    parti = [p for p in _dividi(testo, " -/") if p != ""]
    return _finalizza(parti[-1] if parti else None)


REGOLE = {
    "RIMUOVI_SEPARATORI": _separatori,
    "RIMUOVI_ZERI": _separatori,
    "NESSUNA_REGOLA": _nessuna,
    "TESTO_MAIUSCOLO": _nessuna,
    "SOLO_NUMERI": _numeri,
    "ESTRAI_PRIMA_STRINGA": _estrai_prima,
    "ESTRAI_ULTIMA_STRINGA": _estrai_ultima,
}
REGOLE_NOMI = list(REGOLE)

# A parità di match si preferisce la regola che cambia meno i valori.
ORDINE_REGOLE = [
    "RIMUOVI_SEPARATORI", "RIMUOVI_ZERI", "NESSUNA_REGOLA",
    "TESTO_MAIUSCOLO", "SOLO_NUMERI", "ESTRAI_PRIMA_STRINGA",
    "ESTRAI_ULTIMA_STRINGA",
]
PREFERENZA = {nome: i for i, nome in enumerate(ORDINE_REGOLE)}


def normalizza(valore, regola):
    """Normalizza un codice o un DDT con una delle 7 regole."""
    base = pulito(valore)
    if base is None:
        return None
    funzione = REGOLE.get(
        (regola or "").strip().upper(),
        REGOLE["RIMUOVI_SEPARATORI"],
    )
    return funzione(base)


def chiavi(valori, regola):
    """Insieme delle forme normalizzate, senza valori vuoti."""
    return {
        chiave
        for chiave in (normalizza(v, regola) for v in valori)
        if chiave
    }


# --------------------------------------------------------------
# LETTURA DEI FILE
# --------------------------------------------------------------

def percorso(nome_file):
    base = CARTELLA_LOCALE if MODALITA_LOCALE else cartella
    return f"{base}/{nome_file}"


def copia_in_locale(sorgente, destinazione_fs, destinazione_python):
    if MODALITA_LOCALE:
        if not os.path.exists(sorgente):
            raise FileNotFoundError(f"File non trovato: {sorgente}")
        return sorgente
    if notebookutils.fs.exists(destinazione_fs):
        notebookutils.fs.rm(destinazione_fs)
    if not notebookutils.fs.cp(sorgente, destinazione_fs):
        raise RuntimeError(f"Impossibile copiare in locale: {sorgente}")
    return destinazione_python


def leggi_excel(nome_file, nome_foglio):
    locale_fs = "file:/tmp/prove_" + nome_file.replace(" ", "_")
    locale_py = locale_fs[len("file:"):]
    locale = copia_in_locale(
        percorso(nome_file), locale_fs, locale_py
    )
    return pd.read_excel(
        locale, sheet_name=nome_foglio, dtype=str
    )


print("Regole disponibili:", ", ".join(REGOLE_NOMI))

## [2] Lettura: codici univoci, DDT per fattura, riferimenti

Dagli XML FPR12 si estraggono **solo** i codici articolo (più il tipo virtuale
`RiferimentoAmministrazione`) e i numeri DDT di ogni fattura. I codici vengono resi univoci su
`(CodiceTipoXML, valore pulito)`: nessuna divisione per fattura, nessun conteggio per documento.

Dai carichi e dalle anagrafiche si prendono i valori di riferimento (`CEAN`, `CART`, `CARF` + `CALA`)
e, per ogni fattura, i `NDOGIAB` con il relativo `CFOR`. Il `CFOR` serve solo come etichetta nelle
proposte: non entra nel calcolo dei codici articolo.

In [ ]:
# ==============================================================
# [2] LETTURA: CODICI UNIVOCI, DDT PER FATTURA, RIFERIMENTI
# ==============================================================

def testo(elemento):
    if elemento is not None and elemento.text:
        return elemento.text.strip() or None
    return None


def ceas_da_nome_file(nome_file):
    return PurePosixPath(nome_file).stem


# --------------------------------------------------------------
# 1. CARICHI E ANAGRAFICHE
# --------------------------------------------------------------

df_carichi = leggi_excel(CARICHI_FILE, CARICHI_SHEET)
df_ean = leggi_excel(EAN_FILE, EAN_SHEET)
df_carf = leggi_excel(CARF_FILE, CARF_SHEET)

TABELLE = {
    "CARICHI": df_carichi,
    "EAN": df_ean,
    "CARF": df_carf,
}


def valori_colonna(tabella, colonna):
    df = TABELLE.get(tabella)
    if df is None or colonna not in df.columns:
        return []
    return [v for v in df[colonna].tolist() if pulito(v)]


VALORI_RIFERIMENTO = {
    campo: [
        v
        for tabella, colonna in sorgenti
        for v in valori_colonna(tabella, colonna)
    ]
    for campo, sorgenti in CAMPI_RIFERIMENTO.items()
}

# CFOR: solo etichetta, e DDT del carico per fattura.
MAPPA_CFOR = {}
DDT_CARICO = defaultdict(set)
CFOR_FATTURA = {}

for ceas, numero, ddt, cfor in zip(
    df_carichi["CEAS"],
    df_carichi["NDOGIAA"],
    df_carichi["NDOGIAB"],
    df_carichi["CFOR"],
):
    chiave_doc = (pulito(ceas), pulito(numero))
    if ddt and pulito(ddt):
        DDT_CARICO[chiave_doc].add(ddt)
    cfor_pulito = pulito(cfor)
    if cfor_pulito:
        MAPPA_CFOR.setdefault(chiave_doc, cfor_pulito)
        CFOR_FATTURA.setdefault(chiave_doc, cfor_pulito)


# --------------------------------------------------------------
# 2. XML
# --------------------------------------------------------------

righe_codici = []
righe_ddt = []
errori = []

zip_python = copia_in_locale(
    percorso(ZIP_FILE), zip_locale_fs, zip_locale_python
)

with zipfile.ZipFile(zip_python, mode="r") as archivio:

    danneggiato = archivio.testzip()
    if danneggiato is not None:
        raise zipfile.BadZipFile(
            f"File danneggiato nello ZIP: {danneggiato}"
        )

    for info in archivio.infolist():
        if info.is_dir() or not info.filename.lower().endswith(".xml"):
            continue

        try:
            root = ET.fromstring(archivio.read(info))

            if root.attrib.get("versione", "").upper() != "FPR12":
                continue

            for body in root.findall(".//{*}FatturaElettronicaBody"):
                for allegati in body.findall("{*}Allegati"):
                    body.remove(allegati)

            ceas = ceas_da_nome_file(info.filename)

            for body in root.findall(".//{*}FatturaElettronicaBody"):

                numero = testo(
                    body.find(
                        "./{*}DatiGenerali/"
                        "{*}DatiGeneraliDocumento/{*}Numero"
                    )
                )
                chiave_doc = (pulito(ceas), pulito(numero))

                # Codici articolo: si raccoglie il valore, non la fattura.
                for dettaglio in body.findall(
                    "./{*}DatiBeniServizi/{*}DettaglioLinee"
                ):
                    for blocco in dettaglio.findall(
                        "./{*}CodiceArticolo"
                    ):
                        tipo = testo(
                            blocco.find("./{*}CodiceTipo")
                        ) or "(senza tipo)"
                        valore = testo(
                            blocco.find("./{*}CodiceValore")
                        )
                        if valore:
                            righe_codici.append({
                                "CodiceTipoXML": tipo,
                                "CodiceValoreXML": valore,
                                "ValorePulito": pulito(valore),
                                "ChiaveDocumento": chiave_doc,
                            })

                    riferimento = testo(
                        dettaglio.find(
                            "./{*}RiferimentoAmministrazione"
                        )
                    )
                    if riferimento:
                        righe_codici.append({
                            "CodiceTipoXML": (
                                "RiferimentoAmministrazione"
                            ),
                            "CodiceValoreXML": riferimento,
                            "ValorePulito": pulito(riferimento),
                            "ChiaveDocumento": chiave_doc,
                        })

                # DDT: restano agganciati alla loro fattura.
                for blocco in body.findall(
                    "./{*}DatiGenerali/{*}DatiDDT"
                ):
                    numero_ddt = testo(
                        blocco.find("./{*}NumeroDDT")
                    )
                    if numero_ddt:
                        righe_ddt.append({
                            "ChiaveDocumento": chiave_doc,
                            "NumeroDDT": numero_ddt,
                        })

        except Exception as errore:
            errori.append({
                "File": info.filename,
                "Errore": str(errore),
            })


df_errori = pd.DataFrame(errori, columns=["File", "Errore"])

# Codici UNIVOCI: nessuna colonna di fattura.
df_codici_univoci = (
    pd.DataFrame(
        righe_codici,
        columns=[
            "CodiceTipoXML", "CodiceValoreXML",
            "ValorePulito", "ChiaveDocumento",
        ],
    )
    .drop_duplicates(subset=["CodiceTipoXML", "ValorePulito"])
    .loc[:, [
        "CodiceTipoXML", "CodiceValoreXML", "ValorePulito",
    ]]
    .reset_index(drop=True)
)

# Etichetta CFOR per tipo (serve solo per le proposte).
CFOR_PER_TIPO = defaultdict(set)
for riga in righe_codici:
    cfor = MAPPA_CFOR.get(riga["ChiaveDocumento"])
    if cfor:
        CFOR_PER_TIPO[riga["CodiceTipoXML"]].add(cfor)

# DDT dell'XML per fattura.
DDT_XML = defaultdict(set)
for riga in righe_ddt:
    DDT_XML[riga["ChiaveDocumento"]].add(riga["NumeroDDT"])

# Fatture confrontabili: almeno un DDT su entrambi i lati.
fatture = []
for chiave_doc in sorted(
    set(DDT_XML) & set(DDT_CARICO), key=str
):
    xml = {v for v in DDT_XML[chiave_doc] if pulito(v)}
    carico = {v for v in DDT_CARICO[chiave_doc] if pulito(v)}
    if xml and carico:
        fatture.append({
            "ChiaveDocumento": chiave_doc,
            "XML": xml,
            "Carico": carico,
            "CFOR": CFOR_FATTURA.get(chiave_doc, ""),
        })

print(
    f"Codici univoci: {len(df_codici_univoci):,} "
    f"({df_codici_univoci['CodiceTipoXML'].nunique():,} tipi) - "
    f"fatture confrontabili per DDT: {len(fatture):,}"
)
print(
    "Valori di riferimento:",
    {c: len(v) for c, v in VALORI_RIFERIMENTO.items()},
)
if not df_errori.empty:
    print(f"XML non elaborati: {len(df_errori)}")
    display(df_errori)

## [3] Codici articolo: tutte le combinazioni

Per ogni `CodiceTipoXML` si provano **3 campi × 7 regole XML × 7 regole carico** = 147 combinazioni.
Ogni combinazione normalizza i codici univoci dell'XML con `RegolaXML` e i valori di riferimento con
`RegolaCarico`, poi conta quanti codici combaciano. Nessuna fattura, nessun `CFOR`, nessun peso: solo
conteggi.

`GrigliaCodici` conserva tutte le prove. `ProposteCodici` prende la riga con più match per tipo; a
parità vince la regola meno distruttiva (`Preferenza`). Se nessuna combinazione trova un match, il
campo e le regole restano vuoti: quel tipo va guardato a mano.

Le chiavi normalizzate di 1–2 caratteri non vengono contate come match: con `SOLO_NUMERI` un codice
come `ART-002` diventa `2` e combacerebbe per caso con `C002`. Le regole vengono provate comunque
tutte; si scarta solo l'accoppiamento evidentemente accidentale. Per i DDT questo filtro non serve
(i numeri di bolla corti sono normali).

In [ ]:
# ==============================================================
# [3] CODICI ARTICOLO: TUTTE LE COMBINAZIONI DI REGOLE
# ==============================================================

VALORI_XML = {
    tipo: gruppo["CodiceValoreXML"].tolist()
    for tipo, gruppo in df_codici_univoci.groupby("CodiceTipoXML")
}

# Forme normalizzate calcolate una volta sola.
NORM_XML = {}
for tipo, valori in VALORI_XML.items():
    for regola in REGOLE_NOMI:
        NORM_XML[(tipo, regola)] = [
            normalizza(v, regola) for v in valori
        ]

RIF = {
    (campo, regola): chiavi(valori, regola)
    for campo, valori in VALORI_RIFERIMENTO.items()
    for regola in REGOLE_NOMI
}

righe_griglia = []

for tipo, valori in VALORI_XML.items():
    totale = len(valori)

    for campo in VALORI_RIFERIMENTO:
        for r_xml in REGOLE_NOMI:
            norme = NORM_XML[(tipo, r_xml)]

            for r_carico in REGOLE_NOMI:
                riferimento = RIF[(campo, r_carico)]
                combacianti = [
                    v for v, k in zip(valori, norme)
                    if (
                        k
                        and len(k) >= MIN_LUNGHEZZA_CHIAVE
                        and k in riferimento
                    )
                ]
                match = len(combacianti)

                righe_griglia.append({
                    "CodiceTipoXML": tipo,
                    "CampoCanonico": campo,
                    "RegolaXML": r_xml,
                    "RegolaCarico": r_carico,
                    "CodiciUnivoci": totale,
                    "Match": match,
                    "Quota": round(match / totale, 2) if totale else 0.0,
                    "Preferenza": (
                        PREFERENZA[r_xml] + PREFERENZA[r_carico]
                    ),
                    "EsempiMatch": " | ".join(combacianti[:3]),
                })

df_griglia_codici = pd.DataFrame(
    righe_griglia,
    columns=[
        "CodiceTipoXML", "CampoCanonico", "RegolaXML",
        "RegolaCarico", "CodiciUnivoci", "Match", "Quota",
        "Preferenza", "EsempiMatch",
    ],
)

df_proposte_codici = (
    df_griglia_codici
    .sort_values(
        by=[
            "CodiceTipoXML", "Match", "Preferenza",
            "CampoCanonico", "RegolaXML", "RegolaCarico",
        ],
        ascending=[True, False, True, True, True, True],
    )
    .drop_duplicates("CodiceTipoXML", keep="first")
    .copy()
)

df_proposte_codici["CFOR osservati"] = [
    ", ".join(sorted(CFOR_PER_TIPO.get(tipo, set())))
    for tipo in df_proposte_codici["CodiceTipoXML"]
]
df_proposte_codici["Stato"] = [
    "OK" if m > 0 else "NESSUN MATCH"
    for m in df_proposte_codici["Match"]
]

# Senza match non si propone nulla.
senza_match = df_proposte_codici["Match"] == 0
df_proposte_codici.loc[
    senza_match,
    ["CampoCanonico", "RegolaXML", "RegolaCarico"],
] = ""

COLONNE_PROPOSTE_CODICI = [
    "CodiceTipoXML", "CampoCanonico", "RegolaXML", "RegolaCarico",
    "CodiciUnivoci", "Match", "Quota", "CFOR osservati",
    "Stato", "EsempiMatch",
]
df_proposte_codici = df_proposte_codici[
    COLONNE_PROPOSTE_CODICI
].reset_index(drop=True)

print(
    f"Combinazioni provate: {len(df_griglia_codici):,} "
    f"su {df_codici_univoci['CodiceTipoXML'].nunique():,} tipi"
)
print(
    "Tipi con almeno un match: "
    f"{int((df_proposte_codici['Match'] > 0).sum()):,}"
)

## [4] DDT: tutte le combinazioni, dentro ogni fattura

Per i DDT il confronto resta **dentro la singola fattura**: per ogni coppia `CEAS + numero fattura`
si prendono i `NumeroDDT` dell'XML e gli `NDOGIAB` dei carichi di quella fattura, e si provano le
7 × 7 regole. Un DDT della fattura A non può mai incrociare un DDT della fattura B.

I match si sommano su tutte le fatture, per `CFOR` e complessivamente (`TUTTI`). `ProposteDDT` dà la
coppia di regole migliore per ogni fornitore, `GrigliaDDT` mostra le 49 prove complessive.

In [ ]:
# ==============================================================
# [4] DDT: TUTTE LE COMBINAZIONI, DENTRO OGNI FATTURA
# ==============================================================

COPPIE_REGOLE = [
    (r_xml, r_carico)
    for r_xml in REGOLE_NOMI
    for r_carico in REGOLE_NOMI
]

conteggi = defaultdict(lambda: [0, 0])
FATTURE_PER_CFOR = defaultdict(int)

for fattura in fatture:
    FATTURE_PER_CFOR["TUTTI"] += 1
    if fattura["CFOR"]:
        FATTURE_PER_CFOR[fattura["CFOR"]] += 1

    riferimenti = {
        regola: chiavi(fattura["Carico"], regola)
        for regola in REGOLE_NOMI
    }
    norme = {
        regola: [
            normalizza(v, regola) for v in fattura["XML"]
        ]
        for regola in REGOLE_NOMI
    }

    etichette = ["TUTTI"]
    if fattura["CFOR"]:
        etichette.append(fattura["CFOR"])

    for r_xml, r_carico in COPPIE_REGOLE:
        match = sum(
            1 for chiave in norme[r_xml]
            if chiave in riferimenti[r_carico]
        )

        for etichetta in etichette:
            accumulato = conteggi[(etichetta, r_xml, r_carico)]
            accumulato[0] += match
            accumulato[1] += len(fattura["XML"])


righe_griglia_ddt = []
for (cfor, r_xml, r_carico), (match, totale) in conteggi.items():
    righe_griglia_ddt.append({
        "CFOR": cfor,
        "RegolaDDTXML": r_xml,
        "RegolaDDTCarico": r_carico,
        "Match": match,
        "DDTXML": totale,
        "Quota": round(match / totale, 2) if totale else 0.0,
        "Preferenza": (
            PREFERENZA[r_xml] + PREFERENZA[r_carico]
        ),
        "Fatture": FATTURE_PER_CFOR.get(cfor, 0),
    })

COLONNE_GRIGLIA_DDT = [
    "CFOR", "RegolaDDTXML", "RegolaDDTCarico", "Match",
    "DDTXML", "Quota", "Preferenza", "Fatture",
]
df_griglia_ddt = pd.DataFrame(
    righe_griglia_ddt, columns=COLONNE_GRIGLIA_DDT
)

# Foglio di audit: solo le 49 combinazioni complessive.
df_griglia_ddt_globale = df_griglia_ddt.loc[
    df_griglia_ddt["CFOR"] == "TUTTI"
].reset_index(drop=True)

df_proposte_ddt = (
    df_griglia_ddt
    .sort_values(
        by=[
            "CFOR", "Match", "Preferenza",
            "RegolaDDTXML", "RegolaDDTCarico",
        ],
        ascending=[True, False, True, True, True],
    )
    .drop_duplicates("CFOR", keep="first")
    .reset_index(drop=True)
)

COLONNE_PROPOSTE_DDT = [
    "CFOR", "RegolaDDTXML", "RegolaDDTCarico",
    "Match", "DDTXML", "Quota", "Fatture",
]
df_proposte_ddt = df_proposte_ddt[
    COLONNE_PROPOSTE_DDT
].reset_index(drop=True)

print(
    f"Fatture confrontate: {len(fatture):,} - "
    f"combinazioni per CFOR: {len(df_griglia_ddt):,}"
)

## [5] Proposte ed export

`ProposteCodici`: `CampoCanonico` e `RegolaXML` vanno in `RegoleCodiciXML`, `RegolaCarico` in
`RegoleCodiciCarico` (per ogni `CFOR osservati` + campo). `ProposteDDT`: le due regole vanno in
`RegoleDDT` per il `CFOR` indicato.

Le due griglie servono a guardare le alternative quando i match sono in parità. L'Excel è l'unico
file scritto: **il dizionario non viene mai letto né modificato**.

In [ ]:
# ==============================================================
# [5] PROPOSTE ED EXPORT
# ==============================================================

def mostra(tabella, colonne=None):
    if tabella is None or tabella.empty:
        print("(nessuna riga)")
        return
    display(
        tabella
        if colonne is None
        else tabella[colonne]
    )


print("PROPOSTE CODICI ARTICOLO")
mostra(
    df_proposte_codici,
    [
        "CodiceTipoXML", "CampoCanonico", "RegolaXML",
        "RegolaCarico", "Match", "CodiciUnivoci",
        "CFOR osservati", "Stato",
    ],
)

print("\nPROPOSTE DDT")
mostra(
    df_proposte_ddt,
    [
        "CFOR", "RegolaDDTXML", "RegolaDDTCarico",
        "Match", "DDTXML", "Fatture",
    ],
)


if ESPORTA_PROPOSTE:

    if MODALITA_LOCALE:
        os.makedirs(CARTELLA_LOCALE, exist_ok=True)
        file_export = f"{CARTELLA_LOCALE}/{OUTPUT_FILE}"
    else:
        cartella_export = f"{cartella}/export"
        if not notebookutils.fs.exists(cartella_export):
            notebookutils.fs.mkdirs(cartella_export)
        file_export = f"{cartella_export}/{OUTPUT_FILE}"


    def motore_excel():
        try:
            import xlsxwriter  # noqa: F401
            return "xlsxwriter"
        except ImportError:
            return "openpyxl"


    TABELLE_DA_ESPORTARE = {
        "ProposteCodici": df_proposte_codici,
        "ProposteDDT": df_proposte_ddt,
        "GrigliaCodici": df_griglia_codici,
        "GrigliaDDT": df_griglia_ddt_globale,
    }

    with pd.ExcelWriter(
        file_export, engine=motore_excel()
    ) as writer:
        for nome_foglio, tabella in TABELLE_DA_ESPORTARE.items():
            if tabella is None or tabella.empty:
                print(
                    f"Nessun dato da esportare per: {nome_foglio}"
                )
                continue
            tabella.to_excel(
                writer,
                sheet_name=nome_foglio[:31],
                index=False,
            )
            print(
                f"Esportato {nome_foglio}: "
                f"{len(tabella):,} righe"
            )

    print("\nFile salvato:", file_export)

else:
    print(
        "Export disattivato (ESPORTA_PROPOSTE = False): "
        "le tabelle restano in memoria."
    )

print(
    "\nDizionario codici.xlsx NON letto e NON modificato: "
    "copiare le righe a mano dopo averle verificate."
)